# Shared QA inference pipeline

Publication copy of the shared inference notebook. Execution outputs have been cleared.


# Prepration

## Reproduction configuration

This shared inference notebook was used with the selected final model/checkpoint for each of the six study configurations. Set `QA_MODEL_PATH`, `QA_DATASET_CSV`, and optionally `QA_PREDICTION_OUTPUT` in the environment or edit the fallback paths below for the selected run.

The checkpoint must be the final model/adapter directory for the configuration being evaluated. Model weights and the prepared dataset are not included in this repository.


In [ ]:
# Install required packages for loading the model and inference
! pip install unsloth==2026.5.4
! pip install transformers==4.56.2 trl==0.22.2
! pip install torch==2.10.0+cu128
! pip install xformers==0.0.35
! pip install Triton==3.6.0

In [ ]:
# Import core modules for model loading and inference utilities
from unsloth import FastLanguageModel
import os
import torch
import pandas as pd

In [ ]:
import torch

assert torch.cuda.is_available()

device = torch.cuda.current_device()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)
torch.cuda.synchronize()

# Model loading

In [ ]:
# Load the selected final fine-tuned model/adapter. Set QA_MODEL_PATH for each system.
model_path = os.environ.get("QA_MODEL_PATH", "/kaggle/working/fine_tuned_model")
if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model/checkpoint path not found: {model_path}. Set QA_MODEL_PATH to the selected final model/adapter directory."
    )

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    load_in_4bit=True,
    device_map={'': torch.cuda.current_device()},
)

In [ ]:
# Apply Unsloth's inference optimization for faster decoding (optional for HF models)
FastLanguageModel.for_inference(model)

# Model Querying

In [ ]:
# Load the prepared 3,000-row corpus; set QA_DATASET_CSV for your runtime.
dataset_csv = os.environ.get(
    "QA_DATASET_CSV",
    "/kaggle/input/datasets/lemika0/new-3000-rows-dataset-fixed/finetune_dataset_fixed_3000.csv",
)
if not os.path.isfile(dataset_csv):
    raise FileNotFoundError(f"Prepared dataset not found: {dataset_csv}. Set QA_DATASET_CSV.")
test_dataset = pd.read_csv(dataset_csv)
test_dataset = test_dataset[test_dataset["split"] == "test"].reset_index(drop=True)

# Remove any pre-existing model predictions to avoid overwrite conflicts
columns_from_scoring_part = [
    'predicted_answer',
]
test_dataset.drop(columns_from_scoring_part, axis=1, inplace=True, errors='ignore')
test_dataset

In [ ]:
print(tokenizer.chat_template)

In [ ]:
# Define system prompt to enforce consistent instruction-following behavior during inference
SYSTEM_PROMPT = """You are a helpful AI assistant.

When a user provides context, use only that context to answer their question.
If the question includes multiple sub-questions, answer each one concisely on a new line, in the same order.

Always strive to provide clear, structured, and accurate answers."""

In [ ]:
# Generate model predictions using chat-style prompting and store results
model_answers = []
for index, row in test_dataset.iterrows():
    context = str(row.get("context") or "").strip()
    questions = str(row.get("question") or "").strip()
    
    # Compose chat prompt using system and user messages
    query = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context: {context}\nQuestion(s):\n{questions}" },
    ]

    prompt = tokenizer.apply_chat_template(
    query,
    tokenize=False,
    add_generation_prompt=True,
    )
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
        #padding=True,
        #tokenize=True,
        #add_generation_prompt=True,
    ).to("cuda")

    # Generate model response using constrained decoding settings
    input_length = inputs["input_ids"].shape[1]

    model.eval()
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        use_cache=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    answer = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )
    
    model_answers.append(answer)
    
    # Optional: print raw model output for inspection
    print('----------------------------------------------------------')
    print(answer)

# Append predictions to test DataFrame
test_dataset['predicted_answer'] = model_answers
print('----------------------------------------------------------')
test_dataset

In [ ]:
torch.cuda.synchronize()

peak_allocated = torch.cuda.max_memory_allocated(device)
peak_reserved = torch.cuda.max_memory_reserved(device)

print(f"Peak inference allocated: {peak_allocated / 1024**3:.3f} GB")
print(f"Peak inference reserved:  {peak_reserved / 1024**3:.3f} GB")

# Dataset saving

In [ ]:
import os
prediction_output = os.environ.get(
    "QA_PREDICTION_OUTPUT",
    "/kaggle/working/test_result/sampled_dataset_test_split_only.csv",
)
os.makedirs(os.path.dirname(prediction_output) or ".", exist_ok=True)

# This file contains context, questions, references, and generated answers.
# Keep it in private/local storage; do not commit it to the public repository.
test_dataset.to_csv(prediction_output, index=False)
print(f"Saved prediction file: {prediction_output}")
test_dataset